## Supplementary Figure 6 — Functional convergence of superficial toward deep CA1

Tests the hypothesis that, after CR ablation, **superficial CA1 pyramidal neurons
acquire a more deep-like functional profile** (i.e. the two sub-populations become
less distinguishable).

**Approach (deep-likeness score).** A linear discriminant (LDA) is trained on the
**control** group only to separate deep vs. superficial cells from their functional
features (firing rate, selectivity, sparsity, information content, field size,
stability, bursting index, theta index; right-skewed features log-transformed,
z-scored on the control distribution). The LDA decision value is then applied to
**all** cells as a continuous *deep-likeness score* (higher = more deep-like).
Convergence = the deep-vs-superficial gap in this score shrinks in CR;DTA+.

**Statistics.** Linear mixed model `deepness ~ genotype * layer + (1 | animal)`;
the **genotype x layer interaction** is the formal convergence test.

> **Result (descriptive).** The deep-superficial gap collapses from +0.13 (CR;DTA-)
> to ~0 (CR;DTA+) - superficial cells become as deep-like as deep cells, while deep
> cells are unchanged. The direction supports convergence, but the interaction is
> **not significant at the animal level (p = 0.30, n = 5 mice/group)**; it is
> therefore reported as a consistent trend, not a significant effect.
>
> NB: the deep-likeness score is built from the same features analysed in Fig 4, so
> this is an integrative summary of those changes, not independent evidence.

In [ ]:
import numpy as np
import pandas as pd
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")

FONT_SIZE = 7
TEXT_KWARGS = {"fontsize": FONT_SIZE, "color": "black"}
plt.rcParams.update({
    "font.size": FONT_SIZE, "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "sans-serif"],
    "axes.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})
COL = {"control": "#0000FF", "exp": "#FF0000"}
CONTROL_IDS = ["65165", "65091", "63383", "66539", "65622"]
FEATS = ["Averate_rate", "Selectivity", "Sparsity", "Information_content_rate",
         "Field_size", "stability_ma", "bursting_index", "theta_index"]
LOG_FEATS = ["Averate_rate", "Selectivity", "Information_content_rate", "Field_size"]
TABLE = "/Users/sachuriga/Desktop/Projects/CR_CA1_paper/tables/functional_properties_with_python_measurements_pycirc_infield.pkl"
SAVE = "/Users/sachuriga/Desktop/Projects/CR_CA1_paper/Figures_neuron_report_raw/suppfig6_functional_convergence.pdf"

# ---- load + features ----
df = pd.read_pickle(TABLE)
df = df[(df["buzaki_py_cell_type"] == "pyramidal") & (df["session"] == "A")].copy()
df["animal_id"] = df["animal_id"].astype(str)
df["group_ani"] = np.where(df["animal_id"].isin(CONTROL_IDS), "control", "exp")
for f in FEATS:
    df[f] = pd.to_numeric(df[f], errors="coerce")
for f in LOG_FEATS:
    df[f] = np.log(df[f].where(df[f] > 0))
df = df.dropna(subset=FEATS + ["sub_population"]).copy()

# ---- deep-likeness: LDA trained on CONTROL only ----
ctrl = df[df.group_ani == "control"]
scaler = StandardScaler().fit(ctrl[FEATS])
lda = LDA().fit(scaler.transform(ctrl[FEATS]), (ctrl.sub_population == "deep").astype(int).values)
df["deepness"] = lda.decision_function(scaler.transform(df[FEATS]))

# ---- convergence test: genotype x layer interaction (mixed model) ----
df["g"] = pd.Categorical(df["group_ani"], categories=["control", "exp"])
df["layer"] = pd.Categorical(df["sub_population"], categories=["superficial", "deep"])
m = smf.mixedlm("deepness ~ g*layer", df, groups=df["animal_id"]).fit(reml=True)
ik = [k for k in m.pvalues.index if ":" in k][0]
p_int = m.pvalues[ik]
gap_c = (df[(df.group_ani=="control") & (df.sub_population=="deep")]["deepness"].mean()
         - df[(df.group_ani=="control") & (df.sub_population=="superficial")]["deepness"].mean())
gap_e = (df[(df.group_ani=="exp") & (df.sub_population=="deep")]["deepness"].mean()
         - df[(df.group_ani=="exp") & (df.sub_population=="superficial")]["deepness"].mean())
print(f"deep-sup deepness gap: control={gap_c:+.3f}, exp={gap_e:+.3f}; interaction p={p_int:.3f}")

# ---- figure ----
fig, ax = plt.subplots(figsize=(4.2, 3.4))
groups = [("control", "deep"), ("exp", "deep"), ("control", "superficial"), ("exp", "superficial")]
xpos = {g: i for i, g in enumerate(groups)}
for g in groups:
    sub = df[(df.group_ani == g[0]) & (df.sub_population == g[1])]
    for a in sub.animal_id.unique():
        yy = sub[sub.animal_id == a]["deepness"].values
        ax.scatter(np.random.normal(xpos[g], 0.06, len(yy)), yy, s=3, color=COL[g[0]], alpha=0.22, lw=0, zorder=2)
        ax.scatter(xpos[g], yy.mean(), s=26, color=COL[g[0]], edgecolor="k", lw=0.5, zorder=4)
    am = sub.groupby("animal_id")["deepness"].mean().values
    ax.errorbar(xpos[g], am.mean(), yerr=am.std() / np.sqrt(len(am)), fmt="_",
                color="k", capsize=3, markersize=12, zorder=5, elinewidth=1.0)
ax.axhline(0, ls="--", color="grey", lw=0.6, zorder=0)
ax.set_xticks(range(4))
ax.set_xticklabels(["Deep\nCR;DTA-", "Deep\nCR;DTA+", "Sup\nCR;DTA-", "Sup\nCR;DTA+"], fontsize=6)
ax.set_ylabel("Deep-likeness (LDA score)", **TEXT_KWARGS)
ax.set_title(f"Functional convergence (superficial -> deep-like)\n"
             f"deep-sup gap: {gap_c:+.2f} (ctrl) -> {gap_e:+.2f} (exp);  "
             f"interaction p = {p_int:.2f}", fontsize=6.5)
sns.despine(ax=ax)
fig.tight_layout()
import os
os.makedirs(os.path.dirname(SAVE), exist_ok=True)
fig.savefig(SAVE, transparent=True, bbox_inches="tight")
plt.show()
print("saved ->", SAVE)
